# MIST 单分子 Colab 验证（R3-B04，clean版）. 按顺序跑，每格末尾有 EXPECT 行，对不上就停、贴回来。
单条 B04 PZJSVHARXVPCTD（C3H8FO3P，QDU00009，[M-H]- 15eV Orbitrap）：oracle B=10 formula-only，指纹None，37列。库外修正：只从检索C移除真值别名，绝不从生成结果删真值。
运行时选 T4 GPU。单 B=10 约为历史 20x2 B=100（Kaggle T4 COMPLETE，172-430s/分子）的 1/10 量。单格无输出超 10min 即停；RUNNING 超 60min 无新日志即停；不点任何提交。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive（超时即停）


In [ ]:
!tar xzf /content/drive/MyDrive/mist-single.tar.gz -C /content
!test -d /content/mist-single || mv /content/artifacts/mist-single /content/mist-single
!ls -lh /content/mist-single /content/mist-single/peaks/MSBNK-Qingdao_University-QDU00009.npz
# EXPECT: 看到 single_molecule_input.json fetch_frigid.sh install.sh run_mist_single.py README.md peaks/*.npz（超时即停）


In [ ]:
!cd /content/mist-single && bash install.sh
# EXPECT: 末尾 install done. + cuda True + Tesla T4（pip约数分钟；超时即停）


In [ ]:
import json, hashlib, sys
p = '/content/mist-single'
inp = json.load(open(p + '/single_molecule_input.json'))
assert inp['meta']['holds_heldout'] is False, 'HELDOUT_FLAG'
pk = inp['spectrum']['peak_file']
pp = p + '/' + pk['path_in_pack']
h = hashlib.sha256(open(pp, 'rb').read()).hexdigest()
assert h == pk['sha256'], 'PEAK_SHA_MISMATCH'
import numpy as np
d = np.load(pp); assert sorted(d.files) == ['it', 'mz', 'rel_int'] and len(d['mz']) == 99
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NEED_T4_GPU'
print(torch.cuda.get_device_name(0))
bad = [m for m in sys.modules if any(b in m.lower() for b in ('fpnet', 'dreamsfp', 'dreams_fp', 'dreamms'))]
assert not bad, bad
print('env probe OK', inp['molecule']['key'], inp['molecule']['formula_raw'])
# EXPECT: env probe OK PZJSVHARXVPCTD C3H8FO3P + cuda True + Tesla T4（超时即停）


In [ ]:
!cd /content/mist-single && bash fetch_frigid.sh
# EXPECT: 末尾 FETCH_DONE. + WEIGHT_DOWNLOAD_OK（2.8GB下载；<0.5MB/s且ETA>4h则停；超时即停）


In [ ]:
!cd /content/mist-single && python3 run_mist_single.py
# EXPECT: 末尾 smoke=GEN_OK（或如实SMOKE_NO_VALID）+ wrote single_molecule_result_B10.json；SAFE失败即停；单格无输出超10min停，总超60min脚本自动停并落盘partial（超时即停）


In [ ]:
!cd /content/mist-single/work && ls -lh single_molecule_result_B10.json frigid_commit.txt ckpt_files.txt frigid_tar_list.txt && tar czf mist-single-scores.tar.gz single_molecule_result_B10.json frigid_commit.txt ckpt_files.txt frigid_tar_list.txt && ls -lh mist-single-scores.tar.gz && sha256sum mist-single-scores.tar.gz
# EXPECT: single_molecule_result_B10.json + mist-single-scores.tar.gz（若干MB）+ sha256行。去 Files 面板下载发回（超时即停）。
